# Настройка окружения и загрузка репозитория

In [ ]:
!pip install kenlm jiwer

import sys, os, subprocess, math, heapq
from pathlib import Path
from typing import List, Tuple

import torch
import torchaudio
import kenlm
import jiwer
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

# Для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Клонируем репозиторий задания, если ещё не сделано
REPO_DIR = Path('./asr_assignment2_repo')
if not REPO_DIR.exists():
    !git clone https://github.com/AntonOkhotnikov/ai-talent-hub-itmo-speech-course.git {REPO_DIR}
# Переход в папку assignment2
ASSIGN_DIR = REPO_DIR / 'assignments' / 'assignment2'
sys.path.insert(0, str(ASSIGN_DIR))

# Установка зависимостей (kenlm уже должен быть установлен отдельно, как описано в задании)
!pip install -r {ASSIGN_DIR / 'requirements.txt'} --quiet

In [ ]:
import os
os.chdir(str(ASSIGN_DIR))
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Проверка наличия модели и данных
print("Decoder module:", (ASSIGN_DIR / 'wav2vec2decoder.py').exists())
print("LM 3-gram:", (ASSIGN_DIR / 'lm' / '3-gram.pruned.1e-7.arpa.gz').exists())
print("Examples dir:", (ASSIGN_DIR / 'examples').exists())
print("Data (LibriSpeech test-other):", (ASSIGN_DIR / 'data' / 'librispeech_test_other').exists())
print("Data (Earnings22 test):", (ASSIGN_DIR / 'data' / 'earnings22_test').exists())

In [ ]:
from wav2vec2decoder import Wav2Vec2Decoder, _log_add
# Проверим, что всё импортируется
print("Wav2Vec2Decoder imported successfully")

# Part 1 — CTC Decoding

## 1. Вспомогательные функции для оценки
Функции для вычисления CER/WER на наборах данных.

In [ ]:
import csv

def load_manifest(manifest_path: Path) -> List[Tuple[Path, str]]:
    """
    Читает манифест формата CSV с заголовком path,text.
    Возвращает список (абсолютный путь к .wav, текст транскрипции).
    """
    samples = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # путь может быть относительным от корня репозитория
            rel_path = row['path'].strip()
            # если начинается с 'data/', то путь уже от ASSIGN_DIR (где мы находимся)
            audio_path = Path(rel_path)
            if not audio_path.is_absolute():
                audio_path = Path.cwd() / audio_path  # делаем абсолютным от текущей директории
            transcript = row['text'].strip()
            samples.append((audio_path, transcript))
    return samples

def evaluate_decoder(decoder: Wav2Vec2Decoder,
                     manifest: List[Tuple[Path, str]],
                     method: str) -> Tuple[float, float]:
    """
    Вычисляет WER и CER для заданного метода декодирования на всём манифесте.
    """
    refs = []
    hyps = []
    for audio_path, ref in tqdm(manifest, desc=f"Evaluating {method}"):
        audio_input, sr = torchaudio.load(audio_path)
        assert sr == 16000, f"Sample rate {sr} instead of 16000 for {audio_path}"
        hyp = decoder.decode(audio_input, method=method)
        refs.append(ref)
        hyps.append(hyp)
    wer = jiwer.wer(refs, hyps)
    cer = jiwer.cer(refs, hyps)
    return wer, cer

## Task 1 – Greedy Decoding

In [ ]:
# -------------------- Task 1: Greedy Decoding --------------------
import torch
import torchaudio
from pathlib import Path

# Реализация метода greedy_decode
def greedy_decode(self, logits: torch.Tensor) -> str:
    """
    Perform greedy decoding (find best CTC path).
    """
    # Применяем log_softmax (как требует задание)
    log_probs = torch.log_softmax(logits, dim=-1)
    # Выбираем индексы с максимальной вероятностью
    best_ids = torch.argmax(log_probs, dim=-1)  # (T,)
    # Сжатие CTC: удаляем повторы и blank
    prev = None
    filtered_ids = []
    for idx in best_ids.tolist():
        if idx != self.blank_token_id and idx != prev:
            filtered_ids.append(idx)
        prev = idx
    # Преобразуем в текст
    return self._ids_to_text(filtered_ids)

# Monkey-patching – заменяем метод в классе
Wav2Vec2Decoder.greedy_decode = greedy_decode

# Тест на одном примере из examples/ (быстрая проверка)
decoder_debug = Wav2Vec2Decoder(lm_model_path=None, temperature=1.0)
audio_path = Path('examples/sample1.wav')
audio_input, sr = torchaudio.load(audio_path)
assert sr == 16000
hyp = decoder_debug.decode(audio_input, method="greedy")
print("Example 1 greedy hypothesis:", hyp)

# Полная оценка на librispeech_test_other
# Создаём ещё один декодер (можно использовать тот же)
decoder = Wav2Vec2Decoder(lm_model_path=None, temperature=1.0)
manifest_path = Path('data/librispeech_test_other/manifest.csv')
manifest = load_manifest(manifest_path)
print(f"Loaded {len(manifest)} test samples from LibriSpeech test-other")

wer_g, cer_g = evaluate_decoder(decoder, manifest, method="greedy")
print(f"Greedy Decoding -> WER: {wer_g:.4f} ({wer_g*100:.2f}%), CER: {cer_g:.4f} ({cer_g*100:.2f}%)")
print("Reference WER ≈ 10.4%, CER ≈ 3.5%")

## Task 2 – Beam Search Decoding (без LM)

In [ ]:
# -------------------- Task 2: Beam Search Decoding (without LM) – исправленная версия 2 --------------------
import heapq
from collections import defaultdict

def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
    """
    CTC beam search без языковой модели.
    """
    log_probs = torch.log_softmax(logits, dim=-1)
    T, V = log_probs.shape
    blank = self.blank_token_id  # 0
    forbidden_tokens = {1, 2, 3}  # <s>, </s>, <unk>

    beams = [(0.0, [], blank)]  # (log_prob, compressed_ids, last_token)

    for t in range(T):
        new_beams_dict = defaultdict(lambda: float('-inf'))
        for score, comp_ids, last_tok in beams:
            for c in range(V):
                if c in forbidden_tokens:
                    continue
                # нельзя два одинаковых не-blank подряд
                if c != blank and c == last_tok:
                    continue
                new_score = score + log_probs[t, c].item()

                if c == blank:
                    new_comp_ids = comp_ids
                    new_last = blank
                else:
                    new_comp_ids = comp_ids + [c]
                    new_last = c

                key = (tuple(new_comp_ids), new_last)
                if new_score > new_beams_dict[key]:
                    new_beams_dict[key] = new_score

        # отбираем top beam_width (положительные score, удаляем худший)
        new_beams = []
        for (comp_ids_tuple, last_tok), sc in new_beams_dict.items():
            heapq.heappush(new_beams, (sc, list(comp_ids_tuple), last_tok))
            if len(new_beams) > self.beam_width:
                heapq.heappop(new_beams)  # удаляет минимальный score (худший)

        beams = [(sc, comp_ids, last_tok) for sc, comp_ids, last_tok in new_beams]

    # сортируем финальные гипотезы по убыванию score
    beams.sort(key=lambda x: x[0], reverse=True)

    if return_beams:
        return [(comp_ids, sc) for sc, comp_ids, _ in beams]

    best_ids = beams[0][1]
    return self._ids_to_text(best_ids)

Wav2Vec2Decoder.beam_search_decode = beam_search_decode

# быстрый тест sample1
audio_path = Path('examples/sample1.wav')
audio_input, sr = torchaudio.load(audio_path)
decoder_beam = Wav2Vec2Decoder(lm_model_path=None, temperature=1.0, beam_width=3)
hyp_beam = decoder_beam.decode(audio_input, method="beam")
print("Beam search (width=3) hypothesis:", hyp_beam)

In [ ]:
beam_widths = [1, 3, 10, 50]
results_beam = []
for bw in beam_widths:
    decoder = Wav2Vec2Decoder(lm_model_path=None, temperature=1.0, beam_width=bw)
    manifest_path = Path('data/librispeech_test_other/manifest.csv')
    manifest = load_manifest(manifest_path)
    wer, cer = evaluate_decoder(decoder, manifest, method="beam")
    results_beam.append((bw, wer, cer))
    print(f"Beam width {bw}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

In [ ]:
# -------------------- Task 2: Визуализация --------------------
df_beam = pd.DataFrame(results_beam, columns=["Beam Width", "WER", "CER"])
print("\nBeam width experiments:")
print(df_beam.to_string(formatters={"WER": "{:.4f}".format, "CER": "{:.4f}".format}))

plt.figure(figsize=(8, 5))
plt.plot(df_beam["Beam Width"], df_beam["WER"]*100, marker='o', linewidth=2, markersize=8, label="WER")
plt.plot(df_beam["Beam Width"], df_beam["CER"]*100, marker='s', linewidth=2, markersize=8, label="CER")
plt.xlabel("Beam Width", fontsize=12)
plt.ylabel("Error Rate (%)", fontsize=12)
plt.title("Beam Width vs. Error Rate (LibriSpeech test-other)", fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(df_beam["Beam Width"])
plt.tight_layout()
plt.show()

In [ ]:
import time
import numpy as np

# -------------------- Task 2: Quality vs. Compute Trade-off --------------------
beam_widths = [1, 3, 10, 50]
results_beam = []
time_per_sample = []   # среднее время на один файл (сек)

# Тестовый пример для замера времени (можно взять любой, например sample1)
test_audio, sr = torchaudio.load('examples/sample1.wav')
assert sr == 16000

for bw in beam_widths:
    decoder = Wav2Vec2Decoder(lm_model_path=None, temperature=1.0, beam_width=bw)

    # Замер времени на одном примере (повторим несколько раз для стабильности)
    trials = 5
    t_start = time.time()
    for _ in range(trials):
        _ = decoder.decode(test_audio, method="beam")
    t_elapsed = (time.time() - t_start) / trials

    # Полная оценка WER/CER
    manifest_path = Path('data/librispeech_test_other/manifest.csv')
    manifest = load_manifest(manifest_path)
    wer, cer = evaluate_decoder(decoder, manifest, method="beam")

    results_beam.append((bw, wer, cer))
    time_per_sample.append(t_elapsed)
    print(f"Beam width {bw:2d}: WER={wer*100:.2f}%, CER={cer*100:.2f}%, avg time={t_elapsed:.3f}s")


# Построение графика
fig, ax1 = plt.subplots(figsize=(9, 5))

color_wer = '#e74c3c'
color_cer = '#3498db'
ax1.set_xlabel('Beam Width', fontweight='bold')
ax1.set_ylabel('Error Rate (%)', fontweight='bold')
ax1.plot(beam_widths, [w*100 for _,w,_ in results_beam],
         marker='o', color=color_wer, linewidth=2, markersize=8, label='WER')
ax1.plot(beam_widths, [c*100 for _,_,c in results_beam],
         marker='s', color=color_cer, linewidth=2, markersize=8, label='CER')
ax1.tick_params(axis='y')
ax1.set_xticks(beam_widths)
ax1.grid(True, linestyle='--', alpha=0.4)

# Вторая ось для времени
ax2 = ax1.twinx()
color_time = '#2ecc71'
ax2.set_ylabel('Decoding Time (seconds)', fontweight='bold', color=color_time)
ax2.plot(beam_widths, time_per_sample, marker='D', color=color_time,
         linewidth=2, markersize=8, linestyle='--', label='Time per utterance')
ax2.tick_params(axis='y', labelcolor=color_time)
ax2.set_ylim(bottom=0)

# Легенда
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', frameon=True, fancybox=True)

plt.title('Beam Width: Quality vs. Compute Trade-off\n(LibriSpeech test-other)', fontweight='bold', fontsize=13)
fig.tight_layout()

fig.savefig('beam_width.png', dpi=300, bbox_inches='tight')   # растровый
plt.show()

## Task 3 – Temperature Scaling

In [ ]:
# -------------------- Task 3: Temperature Sweep (Greedy) + Plot --------------------
temperatures = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
results_temp = []
manifest_path = Path('data/librispeech_test_other/manifest.csv')
manifest = load_manifest(manifest_path)

for T in temperatures:
    decoder = Wav2Vec2Decoder(lm_model_path=None, temperature=T)
    wer, cer = evaluate_decoder(decoder, manifest, method="greedy")
    results_temp.append((T, wer, cer))
    print(f"T={T:.1f}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

# Построение графика
df_temp = pd.DataFrame(results_temp, columns=["Temperature", "WER", "CER"])

fig, ax1 = plt.subplots(figsize=(8, 5))

color_wer = '#e74c3c'
color_cer = '#3498db'
ax1.set_xlabel('Temperature', fontweight='bold', fontsize=12)
ax1.set_ylabel('WER / CER (%)', fontweight='bold', fontsize=12)
ax1.plot(df_temp["Temperature"], df_temp["WER"]*100, marker='o', color=color_wer,
         linewidth=2.5, markersize=10, label='WER')
ax1.plot(df_temp["Temperature"], df_temp["CER"]*100, marker='s', color=color_cer,
         linewidth=2.5, markersize=10, label='CER')
ax1.tick_params(axis='y')
ax1.set_xticks(temperatures)
ax1.grid(True, linestyle='--', alpha=0.4)

# Аннотации
for i, (T, w, c) in enumerate(results_temp):
    ax1.annotate(f'{w*100:.2f}%', (T, w*100), textcoords="offset points", xytext=(0,10),
                 ha='center', fontsize=9, color=color_wer)
    ax1.annotate(f'{c*100:.2f}%', (T, c*100), textcoords="offset points", xytext=(0,-15),
                 ha='center', fontsize=9, color=color_cer)

plt.title('Greedy Decoding: Temperature vs. WER/CER\n(LibriSpeech test-other)', fontweight='bold', fontsize=13)
ax1.legend(loc='upper left', frameon=True, fancybox=True)
fig.tight_layout()

# Сохранение
plt.savefig('temperature_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Part 2 — Language Model Integration

## Task 4 – Beam Search with LM Shallow Fusion (3-gram)

In [ ]:
# -------------------- Task 4: Beam search with LM shallow fusion (stable version) --------------------
import heapq
import math
from collections import defaultdict

def beam_search_with_lm(self, logits: torch.Tensor) -> str:
    """
    Beam search decoding with shallow LM fusion.
    score = log_p_acoustic + alpha * log_p_lm + beta * num_words
    """
    if not self.lm_model:
        raise ValueError("KenLM model required for LM shallow fusion")

    log_probs = torch.log_softmax(logits, dim=-1)
    T, V = log_probs.shape
    blank = self.blank_token_id  # 0
    word_delim = self.word_delimiter  # '|'
    space_token = ' '
    forbidden_tokens = {1, 2, 3}  # <s>, </s>, <unk>

    # Луч хранит: (log_prob, token_ids, last_token, text_for_lm, lm_score_so_far, num_words)
    # text_for_lm – строка, которую можно передать в LM (с пробелами вместо '|')
    # lm_score_so_far – кумулятивный LM-скор в натуральных логарифмах
    beams = [(0.0, [], blank, "", 0.0, 0)]

    for t in range(T):
        new_beams_dict = defaultdict(lambda: float('-inf'))
        best_entries = {}

        for score, token_ids, last_tok, lm_text, lm_cum, nwords in beams:
            for c in range(V):
                if c in forbidden_tokens:
                    continue
                if c != blank and c == last_tok:
                    continue

                # Акустический скор
                new_score = score + log_probs[t, c].item()

                if c == blank:
                    # Без изменений текста и LM
                    final_score = new_score + self.alpha * lm_cum + self.beta * nwords
                    key = (tuple(token_ids), blank)
                    if final_score > new_beams_dict[key]:
                        new_beams_dict[key] = final_score
                        best_entries[key] = (token_ids, blank, lm_text, lm_cum, nwords)
                else:
                    new_ids = token_ids + [c]
                    char = self.vocab[c]
                    # Обновляем текст для LM
                    if char == word_delim:
                        new_lm_text = lm_text + space_token
                    else:
                        new_lm_text = lm_text + char.lower()

                    # Вычисляем новый LM-скор строки (в log10 от KenLM, переводим в ln)
                    new_lm_cum = self.lm_model.score(new_lm_text.strip()) * math.log(10)

                    # Считаем количество завершённых слов
                    new_nwords = new_lm_text.strip().count(space_token)

                    final_score = new_score + self.alpha * new_lm_cum + self.beta * new_nwords
                    key = (tuple(new_ids), c)
                    if final_score > new_beams_dict[key]:
                        new_beams_dict[key] = final_score
                        best_entries[key] = (new_ids, c, new_lm_text, new_lm_cum, new_nwords)

        # Отбор top beam_width
        new_beams = []
        for (comp_ids_tuple, last_tok), sc in new_beams_dict.items():
            ids, last, lm_txt, lm_c, nw = best_entries[(comp_ids_tuple, last_tok)]
            heapq.heappush(new_beams, (sc, ids, last, lm_txt, lm_c, nw))
            if len(new_beams) > self.beam_width:
                heapq.heappop(new_beams)

        beams = [(sc, ids, last, lm_txt, lm_c, nw) for sc, ids, last, lm_txt, lm_c, nw in new_beams]

    beams.sort(key=lambda x: x[0], reverse=True)
    best_ids = beams[0][1]
    return self._ids_to_text(best_ids)

Wav2Vec2Decoder.beam_search_with_lm = beam_search_with_lm

# Тест
decoder_lm = Wav2Vec2Decoder(
    lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
    beam_width=10,
    alpha=0.5,
    beta=1.0,
    temperature=1.0
)
hyp_lm = decoder_lm.decode(torchaudio.load('examples/sample1.wav')[0], method="beam_lm")
print("Beam+LM hypothesis:", hyp_lm)

In [ ]:
# -------------------- Task 4: Sweep α, β для shallow fusion --------------------
alphas = [0.01, 0.05, 0.1, 0.5]
betas = [0.0, 0.5, 1.0, 1.5]

results_sf = []
manifest_path = Path('data/librispeech_test_other/manifest.csv')
manifest = load_manifest(manifest_path)

total_combinations = len(alphas) * len(betas)
current = 0

for alpha in alphas:
    for beta in betas:
        current += 1
        print(f"\n[{current}/{total_combinations}] Testing α={alpha:.2f}, β={beta:.1f}...")
        decoder = Wav2Vec2Decoder(
            lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
            beam_width=10,
            alpha=alpha,
            beta=beta,
            temperature=1.0
        )
        wer, cer = evaluate_decoder(decoder, manifest, method="beam_lm")
        results_sf.append((alpha, beta, wer, cer))
        print(f"α={alpha:.2f}, β={beta:.1f}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

# Сохраняем результаты
df_sf = pd.DataFrame(results_sf, columns=["alpha", "beta", "WER", "CER"])
print("\nShallow Fusion Results:")
print(df_sf.to_string(formatters={"WER": "{:.4f}".format, "CER": "{:.4f}".format}))

# Находим лучшую конфигурацию
best_row = df_sf.loc[df_sf['WER'].idxmin()]
print(f"\nBest configuration: α={best_row['alpha']:.2f}, β={best_row['beta']:.1f}")
print(f"WER={best_row['WER']*100:.2f}%, CER={best_row['CER']*100:.2f}%")

In [ ]:
# -------------------- Task 4: Heatmap and Table --------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap WER
pivot_wer = df_sf.pivot(index="beta", columns="alpha", values="WER")
im1 = axes[0].imshow(pivot_wer, cmap='RdYlGn_r', aspect='auto', origin='lower')
axes[0].set_xticks(range(len(alphas)))
axes[0].set_xticklabels([f'{a:.2f}' for a in alphas])
axes[0].set_yticks(range(len(betas)))
axes[0].set_yticklabels([f'{b:.1f}' for b in betas])
axes[0].set_xlabel('α (LM weight)', fontweight='bold', fontsize=11)
axes[0].set_ylabel('β (word bonus)', fontweight='bold', fontsize=11)
axes[0].set_title('WER Heatmap', fontweight='bold', fontsize=13)
plt.colorbar(im1, ax=axes[0], label='WER')
for i in range(len(betas)):
    for j in range(len(alphas)):
        text = axes[0].text(j, i, f'{pivot_wer.iloc[i,j]*100:.2f}%',
                           ha="center", va="center", color="black", fontsize=9)

# Heatmap CER
pivot_cer = df_sf.pivot(index="beta", columns="alpha", values="CER")
im2 = axes[1].imshow(pivot_cer, cmap='RdYlGn_r', aspect='auto', origin='lower')
axes[1].set_xticks(range(len(alphas)))
axes[1].set_xticklabels([f'{a:.2f}' for a in alphas])
axes[1].set_yticks(range(len(betas)))
axes[1].set_yticklabels([f'{b:.1f}' for b in betas])
axes[1].set_xlabel('α (LM weight)', fontweight='bold', fontsize=11)
axes[1].set_ylabel('β (word bonus)', fontweight='bold', fontsize=11)
axes[1].set_title('CER Heatmap', fontweight='bold', fontsize=13)
plt.colorbar(im2, ax=axes[1], label='CER')
for i in range(len(betas)):
    for j in range(len(alphas)):
        text = axes[1].text(j, i, f'{pivot_cer.iloc[i,j]*100:.2f}%',
                           ha="center", va="center", color="black", fontsize=9)

plt.suptitle('Shallow Fusion: α vs β (3-gram LM, LibriSpeech test-other)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('shallow_fusion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 5 – 4-gram LM Evaluation

In [ ]:
# -------------------- Task 5: Download & Evaluate 4-gram LM --------------------
import urllib.request
import gzip
import shutil
from pathlib import Path

# Создаём папку lm, если её нет
lm_dir = Path('lm')
lm_dir.mkdir(exist_ok=True)

# Скачиваем 4-gram LM, если ещё не скачан
url_4gram = "https://www.openslr.org/resources/11/4-gram.arpa.gz"
local_4gram_gz = lm_dir / "4-gram.arpa.gz"
local_4gram = lm_dir / "4-gram.arpa"

if not local_4gram_gz.exists():
    print(f"Downloading 4-gram LM from {url_4gram} ...")
    urllib.request.urlretrieve(url_4gram, local_4gram_gz)
    print("Downloaded.")
else:
    print("4-gram LM already downloaded.")

# Распаковываем, если ещё не распакован (для kenlm нужен .arpa или .gz, он читает gz сам)
# kenlm умеет загружать .gz напрямую, поэтому распаковывать не обязательно.
# Просто убедимся, что файл на месте.
assert local_4gram_gz.exists(), "4-gram LM file missing!"

# Загружаем 4-gram LM и тестируем shallow fusion с лучшими параметрами из Task 4
best_alpha = 0.01
best_beta = 1.0

decoder_4gram = Wav2Vec2Decoder(
    lm_model_path=str(local_4gram_gz),  # kenlm прочитает .gz напрямую
    beam_width=10,
    alpha=best_alpha,
    beta=best_beta,
    temperature=1.0
)

# Загружаем манифест LibriSpeech test-other
manifest_path = Path('data/librispeech_test_other/manifest.csv')
manifest = load_manifest(manifest_path)

wer_4gram, cer_4gram = evaluate_decoder(decoder_4gram, manifest, method="beam_lm")
print(f"4-gram LM: WER={wer_4gram*100:.2f}%, CER={cer_4gram*100:.2f}%")

## Task 6 – Second-pass LM Rescoring

In [ ]:
# -------------------- Task 6: Implement lm_rescore --------------------
import math
from typing import List, Tuple

def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
    """
    Perform second-pass LM rescoring on beam search outputs.
    Args:
        beams: list of (token_ids, log_prob) from beam_search_decode(return_beams=True)
    Returns:
        str: best rescored transcript
    """
    if not self.lm_model:
        raise ValueError("KenLM model required for LM rescoring")

    best_score = float('-inf')
    best_text = ""
    for token_ids, acoustic_logprob in beams:
        # декодируем в текст (уже содержит пробелы вместо '|')
        text = self._ids_to_text(token_ids)
        if not text.strip():
            lm_logprob = 0.0
            num_words = 0
        else:
            # KenLM возвращает log10, переводим в натуральный логарифм
            lm_logprob = self.lm_model.score(text) * math.log(10)
            num_words = len(text.split())
        total_score = acoustic_logprob + self.alpha * lm_logprob + self.beta * num_words
        if total_score > best_score:
            best_score = total_score
            best_text = text
    return best_text

# monkey-patch
Wav2Vec2Decoder.lm_rescore = lm_rescore

# Быстрый тест на одном примере
decoder_test = Wav2Vec2Decoder(
    lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
    beam_width=10,
    alpha=0.1,
    beta=0.5,
    temperature=1.0
)
audio_input, sr = torchaudio.load('examples/sample1.wav')
hyp_rescore = decoder_test.decode(audio_input, method="beam_lm_rescore")
print("Rescore hypothesis:", hyp_rescore)

In [ ]:
# -------------------- Task 6: Sweep α, β для rescoring --------------------
alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]
betas = [0.0, 0.5, 1.0, 1.5]

# Загрузим манифест
manifest_path = Path('data/librispeech_test_other/manifest.csv')
manifest = load_manifest(manifest_path)

# Шаг 1: получим beams для всех примеров с beam_width=10 (без LM)
print("Extracting beams for all utterances...")
all_beams = []  # список списков beams для каждого примера
references = []
for audio_path, ref in tqdm(manifest, desc="Beam extraction"):
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000
    # Создаём временный декодер только для beam_search_decode (без LM)
    temp_decoder = Wav2Vec2Decoder(lm_model_path=None, beam_width=10, temperature=1.0)
    beams = temp_decoder.decode(audio_input, method="beam")  # это вызовет beam_search_decode с return_beams=False
    # Но нам нужен return_beams=True, поэтому обойдём decode()
    inputs = temp_decoder.processor(audio_input, return_tensors="pt", sampling_rate=16000)
    with torch.no_grad():
        logits = temp_decoder.model(inputs.input_values.squeeze(0)).logits[0]
    logits = logits / temp_decoder.temperature
    beams = temp_decoder.beam_search_decode(logits, return_beams=True)
    all_beams.append(beams)
    references.append(ref)

# Шаг 2: для каждой комбинации α,β вычисляем rescore и метрики
results_rescore = []
for alpha in alphas:
    for beta in betas:
        print(f"Rescoring with α={alpha:.2f}, β={beta:.1f}...")
        hyps = []
        # Создаём декодер с LM и нужными параметрами, но модель не грузим – используем только lm_rescore
        rescore_decoder = Wav2Vec2Decoder(
            lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
            beam_width=10,
            alpha=alpha,
            beta=beta,
            temperature=1.0
        )
        for beams_per_utt in tqdm(all_beams, desc=f"Rescoring α={alpha:.2f} β={beta:.1f}", leave=False):
            best_hyp = rescore_decoder.lm_rescore(beams_per_utt)
            hyps.append(best_hyp)
        wer = jiwer.wer(references, hyps)
        cer = jiwer.cer(references, hyps)
        results_rescore.append((alpha, beta, wer, cer))
        print(f"α={alpha:.2f}, β={beta:.1f}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

In [ ]:
# -------------------- Task 6: Visualization & Comparison --------------------
df_rescore = pd.DataFrame(results_rescore, columns=["alpha", "beta", "WER", "CER"])

# Heatmap WER rescoring
pivot_wer_r = df_rescore.pivot(index="beta", columns="alpha", values="WER")
plt.figure(figsize=(8, 5))
plt.imshow(pivot_wer_r, cmap='RdYlGn_r', aspect='auto', origin='lower')
plt.colorbar(label='WER')
plt.xticks(range(len(alphas)), [f'{a:.2f}' for a in alphas])
plt.yticks(range(len(betas)), [f'{b:.1f}' for b in betas])
plt.xlabel('α (LM weight)', fontweight='bold')
plt.ylabel('β (word bonus)', fontweight='bold')
plt.title('Rescoring WER Heatmap (LibriSpeech test-other)', fontweight='bold')
for i in range(len(betas)):
    for j in range(len(alphas)):
        plt.text(j, i, f'{pivot_wer_r.iloc[i,j]*100:.2f}%', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('task6_rescore_heatmap.png', dpi=150)
plt.show()

# Сравнение лучших результатов двух методов
best_sf = df_sf.loc[df_sf['WER'].idxmin()]   # из Task 4
best_rs = df_rescore.loc[df_rescore['WER'].idxmin()]

print("Comparison of best shallow fusion vs rescoring (3-gram LM):")
print(f"Shallow Fusion: WER={best_sf['WER']*100:.2f}%, CER={best_sf['CER']*100:.2f}% (α={best_sf['alpha']:.2f}, β={best_sf['beta']:.1f})")
print(f"Rescoring:      WER={best_rs['WER']*100:.2f}%, CER={best_rs['CER']*100:.2f}% (α={best_rs['alpha']:.2f}, β={best_rs['beta']:.1f})")

In [ ]:
# -------------------- Task 6: Qualitative Examples (исправленный) --------------------
interesting_indices = []
for idx, (beams_per_utt, ref) in enumerate(zip(all_beams, references)):
    beam_text = rescore_decoder._ids_to_text(beams_per_utt[0][0]) if beams_per_utt else ""
    # Загружаем аудио
    audio_path, _ = manifest[idx]
    audio_input, sr = torchaudio.load(audio_path)
    # SF
    sf_decoder = Wav2Vec2Decoder(lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                                 beam_width=10, alpha=best_sf['alpha'], beta=best_sf['beta'], temperature=1.0)
    sf_text = sf_decoder.decode(audio_input, method="beam_lm")
    # RS
    rs_decoder = Wav2Vec2Decoder(lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                                 beam_width=10, alpha=best_rs['alpha'], beta=best_rs['beta'], temperature=1.0)
    rs_text = rs_decoder.lm_rescore(beams_per_utt)

    if beam_text != sf_text or beam_text != rs_text or sf_text != rs_text:
        interesting_indices.append(idx)
        if len(interesting_indices) >= 10:
            break

# Вывод примеров
for idx in interesting_indices:
    audio_path, ref = manifest[idx]
    audio_input, _ = torchaudio.load(audio_path)
    beams = all_beams[idx]
    beam_text = rescore_decoder._ids_to_text(beams[0][0])
    sf_text = sf_decoder.decode(audio_input, method="beam_lm")
    rs_text = rs_decoder.lm_rescore(beams)
    print(f"REF : {ref}")
    print(f"BEAM: {beam_text}")
    print(f"SF  : {sf_text}")
    print(f"RS  : {rs_text}")
    if sf_text == ref:
        print("  SF  ✓ corrected")
    elif sf_text != beam_text:
        print("  SF  changed")
    if rs_text == ref:
        print("  RS  ✓ corrected")
    elif rs_text != beam_text:
        print("  RS  changed")
    print("-" * 60)

In [ ]:
# -------------------- Task 6: Stability to α --------------------
beta_fixed = best_sf['beta']  # или 0.5
alphas_range = alphas
wer_sf_fixed = []
wer_rs_fixed = []
for a in alphas_range:
    # SF
    dec_sf = Wav2Vec2Decoder(lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                            beam_width=10, alpha=a, beta=beta_fixed, temperature=1.0)
    wer_sf, _ = evaluate_decoder(dec_sf, manifest, method="beam_lm")
    wer_sf_fixed.append(wer_sf)
    # RS
    dec_rs = Wav2Vec2Decoder(lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                            beam_width=10, alpha=a, beta=beta_fixed, temperature=1.0)
    hyps = []
    for beams_per_utt in all_beams:
        hyps.append(dec_rs.lm_rescore(beams_per_utt))
    wer_rs = jiwer.wer(references, hyps)
    wer_rs_fixed.append(wer_rs)

plt.figure(figsize=(8, 5))
plt.plot(alphas_range, [w*100 for w in wer_sf_fixed], marker='o', label='Shallow Fusion')
plt.plot(alphas_range, [w*100 for w in wer_rs_fixed], marker='s', label='Rescoring')
plt.xlabel('α (LM weight)')
plt.ylabel('WER (%)')
plt.title(f'Stability to α (β={beta_fixed})')
plt.legend()
plt.grid(True)
plt.savefig('task6_stability.png', dpi=150)
plt.show()

## Task 7 – Out-of-Domain Evaluation (Earnings22)

In [ ]:
# -------------------- Task 7: Out-of-Domain Evaluation (Earnings22) --------------------
# Загружаем манифест Earnings22 test
earnings_manifest_path = Path('data/earnings22_test/manifest.csv')
earnings_manifest = load_manifest(earnings_manifest_path)
print(f"Loaded {len(earnings_manifest)} samples from Earnings22 test")

# Лучшие параметры из Task 4 (SF) и Task 6 (rescoring) – замените на свои, если отличаются
best_sf_alpha = 0.1    # <-- подставьте из Task 4
best_sf_beta = 0.5     # <-- подставьте из Task 4
best_rs_alpha = 0.1    # <-- подставьте из Task 6
best_rs_beta = 0.5     # <-- подставьте из Task 6
beam_width = 10

# Словарь конфигураций
configs = {
    "greedy": {"lm": None, "alpha": 0, "beta": 0, "temp": 1.0, "bw": 1},
    "beam": {"lm": None, "alpha": 0, "beta": 0, "temp": 1.0, "bw": beam_width},
    "beam_lm": {"lm": "lm/3-gram.pruned.1e-7.arpa.gz", "alpha": best_sf_alpha, "beta": best_sf_beta, "temp": 1.0, "bw": beam_width},
    "beam_lm_rescore": {"lm": "lm/3-gram.pruned.1e-7.arpa.gz", "alpha": best_rs_alpha, "beta": best_rs_beta, "temp": 1.0, "bw": beam_width}
}

results_earnings = {}
for method, cfg in configs.items():
    decoder = Wav2Vec2Decoder(
        lm_model_path=cfg["lm"],
        beam_width=cfg["bw"],
        alpha=cfg["alpha"],
        beta=cfg["beta"],
        temperature=cfg["temp"]
    )
    wer, cer = evaluate_decoder(decoder, earnings_manifest, method=method)
    results_earnings[method] = (wer, cer)
    print(f"{method:20s}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

### 7a – сравнение всех методов

In [ ]:
# -------------------- Task 7a: Comparison Table --------------------
# Вставьте свои значения для LibriSpeech test-other (из Tasks 1,2,4,6)
libri = {
    "greedy": (0.1122, 0.0381),
    "beam": (0.099, 0.034),      # примерные, замените на свои
    "beam_lm": (0.097, 0.034),   # из Task 4 best
    "beam_lm_rescore": (0.096, 0.033)  # из Task 6 best
}

methods_order = ["greedy", "beam", "beam_lm", "beam_lm_rescore"]
table_data = []
for m in methods_order:
    l_wer, l_cer = libri[m]
    e_wer, e_cer = results_earnings[m]
    table_data.append([m, l_wer, l_cer, e_wer, e_cer])

df_comp = pd.DataFrame(table_data, columns=["Method", "LibriSpeech WER", "LibriSpeech CER", "Earnings22 WER", "Earnings22 CER"])
df_comp.set_index("Method", inplace=True)

# Форматирование
format_dict = {col: "{:.2%}".format for col in df_comp.columns}
print("\nComparison of all methods on In-Domain vs Out-of-Domain:")
print(df_comp.to_string(formatters=format_dict))

### 7b – Temperature Sweep на Earnings22 с LM

In [ ]:
# -------------------- Task 7b: Temperature Sweep (Earnings22) --------------------
temperatures = [0.5, 1.0, 1.5, 2.0]
greedy_wer_temp = []
sf_wer_temp = []

for T in temperatures:
    # Greedy
    dec_g = Wav2Vec2Decoder(lm_model_path=None, temperature=T)
    wer_g, _ = evaluate_decoder(dec_g, earnings_manifest, method="greedy")
    greedy_wer_temp.append(wer_g)

    # Shallow fusion (с лучшими alpha, beta)
    dec_sf = Wav2Vec2Decoder(
        lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
        beam_width=beam_width, alpha=best_sf_alpha, beta=best_sf_beta, temperature=T
    )
    wer_sf, _ = evaluate_decoder(dec_sf, earnings_manifest, method="beam_lm")
    sf_wer_temp.append(wer_sf)

    print(f"T={T:.1f}: Greedy WER={wer_g*100:.2f}%, SF WER={wer_sf*100:.2f}%")

# График WER vs T
plt.figure(figsize=(8, 5))
plt.plot(temperatures, [w*100 for w in greedy_wer_temp], marker='o', linewidth=2, label='Greedy (Earnings22)')
plt.plot(temperatures, [w*100 for w in sf_wer_temp], marker='s', linewidth=2, label='Shallow Fusion (Earnings22)')
# Для сравнения можно добавить кривую greedy на LibriSpeech из Task 3 (раскомментируйте и подставьте свои значения)
# libri_greedy_wer = [0.112, 0.111, 0.112, 0.114, 0.118, 0.125]  # пример для T = [0.5,0.8,1.0,1.2,1.5,2.0]
# plt.plot(temperatures, libri_greedy_wer, marker='^', linestyle='--', label='Greedy (LibriSpeech)')
plt.xlabel('Temperature', fontweight='bold')
plt.ylabel('WER (%)', fontweight='bold')
plt.title('Effect of Temperature on Out-of-Domain Decoding (Earnings22)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(temperatures)
plt.tight_layout()
plt.savefig('task7b_temperature_earnings22.png', dpi=150)
plt.show()

## Task 8 – Обучение финансовой KenLM

In [ ]:
# -------------------- Task 8.1: Build KenLM tools --------------------
import subprocess, os
from pathlib import Path

# Установим пакеты, если ещё не (в Colab обычно уже есть)
!apt-get update -qq && apt-get install -y -qq cmake libboost-all-dev 2>&1 | tail -5

# Клонируем kenlm во временную папку и собираем
kenlm_build_dir = Path("/tmp/kenlm_build")
if not kenlm_build_dir.exists():
    !git clone --depth=1 https://github.com/kpu/kenlm /tmp/kenlm_build
    !mkdir -p /tmp/kenlm_build/build && cd /tmp/kenlm_build/build && cmake .. && make -j4 lmplz build_binary
else:
    print("KenLM build directory already exists, skipping compilation.")

# Проверяем наличие утилит
lmplz_path = "/tmp/kenlm_build/build/bin/lmplz"
build_binary_path = "/tmp/kenlm_build/build/bin/build_binary"
print("lmplz exists:", os.path.isfile(lmplz_path))
print("build_binary exists:", os.path.isfile(build_binary_path))

In [ ]:
# -------------------- Task 8.2: Train 3-gram LM --------------------
# Убедимся, что lm/ существует
lm_dir = Path('lm')
lm_dir.mkdir(exist_ok=True)

corpus_path = "data/earnings22_train/corpus.txt"
output_arpa = "/tmp/financial-3gram.arpa"
output_gz = lm_dir / "financial-3gram.arpa.gz"

# Обучаем 3-gram модель (займёт несколько минут)
print("Training 3-gram KenLM on financial corpus...")
!{lmplz_path} -o 3 --discount_fallback < {corpus_path} > {output_arpa}
print("Training completed.")

# Сжимаем в gzip
!gzip -c {output_arpa} > {output_gz}
print(f"Model saved to {output_gz}")

In [ ]:
# -------------------- Task 8.3: Verify LM --------------------
import kenlm
fin_lm = kenlm.Model(str(output_gz))
print("Vocabulary size:", fin_lm.order)  # не совсем размер словаря, но показывает n-gram порядок
# Протестируем на финансовой фразе
test_sentences = ["revenue increased by ten percent", "operating surplus is a non gaap financial measure"]
for s in test_sentences:
    score = fin_lm.score(s)
    print(f"'{s}' -> log10 prob: {score:.3f}")

## Task 9 – Сравнение LMs на обоих доменах

In [ ]:
# -------------------- Task 9: LibriSpeech evaluation --------------------
manifest_ls = load_manifest(Path('data/librispeech_test_other/manifest.csv'))

# Функция для быстрой оценки
def evaluate_method(lm_path, method, alpha, beta, manifest):
    decoder = Wav2Vec2Decoder(
        lm_model_path=lm_path,
        beam_width=beam_width,
        alpha=alpha,
        beta=beta,
        temperature=1.0
    )
    return evaluate_decoder(decoder, manifest, method=method)

# LibriSpeech: SF и rescoring с обеими LM
ls_results = {}
# LibriSpeech LM
ls_results['SF_libri'] = evaluate_method(lm_libri, 'beam_lm', best_sf_alpha, best_sf_beta, manifest_ls)
ls_results['RS_libri'] = evaluate_method(lm_libri, 'beam_lm_rescore', best_rs_alpha, best_rs_beta, manifest_ls)
# Financial LM
ls_results['SF_fin'] = evaluate_method(lm_fin, 'beam_lm', best_sf_alpha, best_sf_beta, manifest_ls)
ls_results['RS_fin'] = evaluate_method(lm_fin, 'beam_lm_rescore', best_rs_alpha, best_rs_beta, manifest_ls)

print("LibriSpeech test-other:")
for k, (wer, cer) in ls_results.items():
    print(f"  {k}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

In [ ]:
# -------------------- Task 9: Earnings22 evaluation --------------------
manifest_earn = load_manifest(Path('data/earnings22_test/manifest.csv'))

earn_results = {}
# LibriSpeech LM
earn_results['SF_libri'] = evaluate_method(lm_libri, 'beam_lm', best_sf_alpha, best_sf_beta, manifest_earn)
earn_results['RS_libri'] = evaluate_method(lm_libri, 'beam_lm_rescore', best_rs_alpha, best_rs_beta, manifest_earn)
# Financial LM
earn_results['SF_fin'] = evaluate_method(lm_fin, 'beam_lm', best_sf_alpha, best_sf_beta, manifest_earn)
earn_results['RS_fin'] = evaluate_method(lm_fin, 'beam_lm_rescore', best_rs_alpha, best_rs_beta, manifest_earn)

print("Earnings22 test:")
for k, (wer, cer) in earn_results.items():
    print(f"  {k}: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

In [ ]:
# -------------------- Task 9: Summary Table --------------------
data = []
for method_lm in ['SF_libri', 'RS_libri', 'SF_fin', 'RS_fin']:
    ls_wer, ls_cer = ls_results[method_lm]
    earn_wer, earn_cer = earn_results[method_lm]
    data.append([method_lm, ls_wer, ls_cer, earn_wer, earn_cer])

df_final = pd.DataFrame(data, columns=['Method+LM', 'Libri WER', 'Libri CER', 'Earnings WER', 'Earnings CER'])
df_final.set_index('Method+LM', inplace=True)
print("Comparison of LMs across domains:")
print(df_final.to_string(formatters={col: '{:.2%}'.format for col in df_final.columns}))

In [ ]:
# -------------------- Task 9: Bar Chart --------------------
labels = df_final.index.tolist()
libri_wer = df_final['Libri WER'].values * 100
earn_wer = df_final['Earnings WER'].values * 100

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, libri_wer, width, label='LibriSpeech WER', color='#3498db')
bars2 = ax.bar(x + width/2, earn_wer, width, label='Earnings22 WER', color='#e74c3c')

ax.set_ylabel('WER (%)', fontweight='bold')
ax.set_title('WER by LM and Domain', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.legend()

# Добавим значения над столбцами
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('task9_lm_domain_comparison.png', dpi=150)
plt.show()